# Response surfaces and steepest ascent in two factors

**Source worksheets:** [yint.org/w10](https://yint.org/w10) and [yint.org/w11](https://yint.org/w11) - weeks 10 and 11 of the applied DoE short course.

Module 6 finished the 1-D version of optimization: pick a factor, take
small steps, fit a linear model, swap to a quadratic when the surface
curves, predict the peak, confirm.  Module 7 generalizes that loop to
**two factors**, using the classical response-surface trio of
*factorial points*, a *center point* to detect curvature, and *axial
points* to fit a quadratic.

This module walks the loop end-to-end on a 2-factor process whose
response surface is quadratic in both factors plus an interaction.

## Q1 from w11 - picking sensible coded ranges

The ecommerce team has narrowed eight factors down to two:

- **A = items in the menu/navigation bar** (1 to 10)
- **B = products shown per web page** (1 to 50)

The worksheet asks: what are reasonable starting values for the ``-1``
and ``+1`` levels?

The principle: pick a *narrow* initial range that you can change
safely on the live site, centered near your current operating point.
You can always widen later.  A *wide* range invites the linear model
to badly approximate a curving surface.

## Q2 - a 2-D optimization, end to end

Below is a concrete 2-factor system.  Think of two coded process
inputs ``x1`` and ``x2`` (temperature and reaction time, for example)
driving some yield-like response ``y``.  The hidden surface is

```
y(x1, x2) = 70 + 8*x1 + 4*x2
            - 3*x1**2 - 2*x2**2
            - 2*x1*x2
            + noise(0, 0.5)
```

The true peak sits at ``(x1, x2) = (1.22, 0.33)`` with response
``y = 75.6`` - but the optimizer does not know that.  Only the function
call ``observe(x1, x2)`` is allowed, and each call returns a *noisy*
measurement.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

from process_improve.experiments import c, gather, lm, predict

pio.renderers.default = "notebook_connected"

rng = np.random.default_rng(seed=7)


def observe(x1: float, x2: float) -> float:
    """Return a noisy measurement of the hidden 2-D surface."""
    truth = (70 + 8 * x1 + 4 * x2
             - 3 * x1 ** 2 - 2 * x2 ** 2
             - 2 * x1 * x2)
    return float(truth + rng.normal(scale=0.5))

### Step 1: 2x2 factorial plus one center run

Spend five runs first: the four corners of a 2x2 plus the center.
Fit a linear-plus-interaction model and check whether the center's
measured response matches the model's prediction at ``(0, 0)``.  A
big mismatch is the surface's way of telling you it is curving.

In [ ]:
x1 = c(-1, +1, -1, +1, 0, name="x1")
x2 = c(-1, -1, +1, +1, 0, name="x2")
y_obs = [observe(a, b) for a, b in zip([-1, 1, -1, 1, 0], [-1, -1, 1, 1, 0], strict=True)]
y = c(*y_obs, name="y")
step1 = gather(x1=x1, x2=x2, y=y)
step1

In [ ]:
m_lin = lm("y ~ x1 + x2 + x1:x2", step1)
print("Linear-with-interaction fit on 2x2+center:")
print(m_lin.get_parameters(drop_intercept=False).to_string())
centre_pred = float(predict(m_lin, x1=0, x2=0).iloc[0])
print()
print(f"Predicted center y at (0, 0): {centre_pred:.2f}")
print(f"Observed center y at (0, 0):  {y_obs[4]:.2f}")
print(f"Difference: {y_obs[4] - centre_pred:+.2f}")

### Step 2: axial points -> central composite design (CCD)

A central composite design adds four axial runs at
``(+/-alpha, 0)`` and ``(0, +/-alpha)``.  Rotatable choice is
``alpha = sqrt(2) ~= 1.414`` for two factors.

Combining the original five runs with the four new ones gives a 9-run
CCD that supports a full quadratic model
``y = b_0 + b_1 x_1 + b_2 x_2 + b_11 x_1^2 + b_22 x_2^2 + b_12 x_1 x_2``.

In [ ]:
alpha = np.sqrt(2)
x1_axial = [-alpha, alpha, 0, 0]
x2_axial = [0, 0, -alpha, alpha]
y_axial = [observe(a, b) for a, b in zip(x1_axial, x2_axial, strict=True)]

x1_ccd = c(-1, +1, -1, +1, 0, *x1_axial, name="x1")
x2_ccd = c(-1, -1, +1, +1, 0, *x2_axial, name="x2")
y_ccd  = c(*y_obs, *y_axial, name="y")
ccd = gather(x1=x1_ccd, x2=x2_ccd, y=y_ccd)
ccd

In [ ]:
m_quad = lm("y ~ x1 + x2 + I(x1**2) + I(x2**2) + x1:x2", ccd)
print("Quadratic fit on the 9-run CCD:")
print(m_quad.get_parameters(drop_intercept=False).to_string())

### Step 3: find the predicted optimum

For the quadratic ``y = b_0 + b_1 x_1 + b_2 x_2 + b_11 x_1^2 + b_22 x_2^2 + b_12 x_1 x_2``,
the stationary point satisfies the gradient equation
``[2 b_11, b_12; b_12, 2 b_22] * x = -[b_1; b_2]``.
A two-by-two solve gives the predicted peak in closed form.

In [ ]:
p = m_quad.get_parameters(drop_intercept=False)
A = np.array([[2 * p["I(x1 ** 2)"], p["x1:x2"]],
              [p["x1:x2"], 2 * p["I(x2 ** 2)"]]])
b = np.array([-p["x1"], -p["x2"]])
x_star = np.linalg.solve(A, b)
y_star = float(predict(m_quad, x1=x_star[0], x2=x_star[1]).iloc[0])
print(f"Predicted optimum: x1 = {x_star[0]:.3f}, x2 = {x_star[1]:.3f}")
print(f"Predicted response at the optimum: y = {y_star:.2f}")

In [ ]:
# Confirm the optimum with one more run.

y_confirm = observe(x_star[0], x_star[1])
print(f"Confirmation run at the predicted optimum: y = {y_confirm:.2f}")
print(f"Predicted: {y_star:.2f} | Observed: {y_confirm:.2f} | Diff: {y_confirm - y_star:+.2f}")

In [ ]:
# Contour plot of the fitted surface, overlaid with the runs.

grid = np.linspace(-1.8, 1.8, 50)
G1, G2 = np.meshgrid(grid, grid)
Z = (p["Intercept"]
     + p["x1"] * G1 + p["x2"] * G2
     + p["I(x1 ** 2)"] * G1 ** 2
     + p["I(x2 ** 2)"] * G2 ** 2
     + p["x1:x2"] * G1 * G2)

fig = go.Figure()
fig.add_trace(go.Contour(x=grid, y=grid, z=Z, contours_coloring="lines",
                         line_width=1, showscale=False))
fig.add_trace(go.Scatter(x=ccd["x1"], y=ccd["x2"], mode="markers",
                         marker={"size": 9, "color": "black"},
                         name="CCD runs"))
fig.add_trace(go.Scatter(x=[x_star[0]], y=[x_star[1]], mode="markers",
                         marker={"size": 14, "symbol": "star", "color": "red"},
                         name="Predicted optimum"))
fig.update_layout(width=560, height=480,
                  xaxis_title="x1 (coded)", yaxis_title="x2 (coded)",
                  title="Fitted response surface with predicted optimum")
fig

## Wrap-up

The 2-D response-surface workflow in one paragraph:

1. **Factorial + center point** (5 runs): if the center disagrees with
   the linear fit, you have curvature.
2. **Add axial points** to make a CCD (9 runs total).  Fit a full
   quadratic.
3. **Solve the gradient = 0 equation** for the predicted peak.
4. **Confirm** with one or two runs at and near the predicted peak.
5. **Sensitivity check**: explore a small region around the optimum
   to confirm no nearby point beats it.

**Next:** Module 8 wraps up the whole course - we collect the
vocabulary in one place, map it back to the modules, and point to
the parts of `process_improve` that handle each step.